[![Abrir no Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lalvim/DataMining/blob/main/notebooks/unidade_05/05_03_classificadores_bayesianos.ipynb)

# Unidade V — Classificação e Regressão

## Classificadores bayesianos

**Carga estimada:** 1,5 hora  
**Pré-requisitos:** probabilidades condicionais e classificação.

> **Pergunta norteadora:** como atualizar a probabilidade de uma classe a partir das evidências?

Este notebook apresenta Naive Bayes, independência condicional, suavização de Laplace e um cálculo manual completo.

## Objetivos de aprendizagem

- interpretar probabilidade prévia, verossimilhança e posterior;
- explicar independência condicional;
- transformar contagens em probabilidades condicionais;
- aplicar suavização de Laplace;
- calcular escores e posteriores;
- reconhecer limitações de calibração e correlação.

In [1]:
import numpy as np
import pandas as pd

## 1. Naive Bayes: atualizar uma crença com evidências

Considere uma classe $C$ e um objeto descrito por atributos $\mathbf{x}$. O teorema de Bayes escreve:

$$
P(C\mid\mathbf{x})
=
\frac{P(\mathbf{x}\mid C)P(C)}{P(\mathbf{x})}.
$$

- $P(C)$ é a **probabilidade prévia**: frequência da classe antes de observar o novo objeto;
- $P(\mathbf{x}\mid C)$ é a **verossimilhança**: compatibilidade dos atributos observados com a classe;
- $P(C\mid\mathbf{x})$ é a **probabilidade posterior**: crença atualizada depois das evidências;
- $P(\mathbf{x})$ é comum a todas as classes do mesmo objeto.

Para escolher entre classes, basta comparar os numeradores:

$$
\operatorname{escore}(C)=P(C)P(\mathbf{x}\mid C).
$$

A classe com maior escore terá também a maior posterior.

### Por que o método é chamado de “ingênuo”?

Se $\mathbf{x}=(x_1,x_2,\ldots,x_p)$, o Naive Bayes simplifica a verossimilhança:

$$
P(\mathbf{x}\mid C)
\approx
\prod_{j=1}^{p}P(x_j\mid C).
$$

Ele trata os atributos como **condicionalmente independentes dentro de cada classe**. Isso permite estimar várias probabilidades pequenas em vez de uma combinação para cada perfil possível.

A suposição frequentemente é imperfeita. Por exemplo, “muitos chamados” e “texto urgente” podem continuar relacionados mesmo entre chamados prioritários. Nessa situação, as duas evidências podem ser contadas como se trouxessem informações separadas. Ainda assim, o método costuma ser uma referência rápida e útil, especialmente em dados de texto e atributos de contagem.

## 2. Exemplo manual: prioridade de chamados

A tabela sintética contém 12 chamados já classificados. Usaremos três atributos categóricos para decidir se um novo chamado deve ser tratado como prioritário.

In [2]:
chamados = pd.DataFrame({
    "canal": [
        "app", "app", "web", "app", "web", "app",
        "web", "web", "app", "web", "app", "web",
    ],
    "texto_urgente": [
        "sim", "sim", "sim", "nao", "sim", "sim",
        "nao", "nao", "nao", "sim", "nao", "nao",
    ],
    "cliente_premium": [
        "sim", "nao", "sim", "sim", "nao", "sim",
        "nao", "sim", "nao", "nao", "sim", "nao",
    ],
    "prioritaria": [
        "sim", "sim", "sim", "sim", "sim", "sim",
        "nao", "nao", "nao", "nao", "nao", "nao",
    ],
})
chamados

,canal,texto_urgente,cliente_premium,prioritaria
0,app,sim,sim,sim
1,app,sim,nao,sim
2,web,sim,sim,sim
3,app,nao,sim,sim
4,web,sim,nao,sim
5,app,sim,sim,sim
6,web,nao,nao,nao
7,web,nao,sim,nao
8,app,nao,nao,nao
9,web,sim,nao,nao


### Um novo chamado

Classificaremos:

$$
\mathbf{x}=(\text{canal=app},\ \text{texto urgente=sim},\ \text{cliente premium=sim}).
$$

Para abreviar a notação, escreveremos:

- $C=\text{sim}$ para a classe **prioritária**;
- $C=\text{não}$ para a classe **não prioritária**.

Há seis chamados em cada classe. Portanto, as probabilidades prévias são:

$$
P(C=\text{sim})=\frac{N_{C=\text{sim}}}{N}
=\frac{6}{12}=0{,}5,
$$

$$
P(C=\text{não})=\frac{N_{C=\text{não}}}{N}
=\frac{6}{12}=0{,}5.
$$

Aqui, $N$ é a quantidade total de chamados e $N_{C=c}$ é a quantidade de chamados da classe $c$.

### Como transformar uma contagem em probabilidade condicional

Usaremos **suavização de Laplace**. Para o valor $v$ de um atributo $x_j$ e uma classe $c$:

$$
P(x_j=v\mid C=c)
=
\frac{N_{x_j=v,\,C=c}+1}{N_{C=c}+K_j}.
$$

Nessa expressão:

- $N_{x_j=v,\,C=c}$ conta os casos da classe $c$ que possuem $x_j=v$;
- $N_{C=c}=6$ é o total de casos da classe;
- $K_j=2$ é o número de categorias possíveis do atributo neste exemplo;
- o $+1$ no numerador e o $+K_j$ no denominador realizam a suavização.

Assim, cada denominador será $6+2=8$. Somar 1 evita que uma evidência ainda não observada em uma classe anule todo o produto do Naive Bayes.

In [3]:
novo_chamado = {
    "canal": "app",
    "texto_urgente": "sim",
    "cliente_premium": "sim",
}
classes = ["sim", "nao"]
escores = {}

for classe in classes:
    dados_classe = chamados[chamados["prioritaria"] == classe]
    previa = len(dados_classe) / len(chamados)
    fatores = {}
    escore = previa

    for atributo, valor in novo_chamado.items():
        contagem = (dados_classe[atributo] == valor).sum()
        numero_categorias = chamados[atributo].nunique()
        probabilidade = (contagem + 1) / (
            len(dados_classe) + numero_categorias
        )
        fatores[f"P({atributo}={valor} | C)"] = probabilidade
        escore *= probabilidade

    escores[classe] = {
        "P(C)": previa,
        **fatores,
        "escore(C)": escore,
    }

tabela_escores = pd.DataFrame(escores).T
tabela_escores.index.name = "C = prioritária"
soma_escores = tabela_escores["escore(C)"].sum()
tabela_escores["P(C | x)"] = tabela_escores["escore(C)"] / soma_escores

assert np.isclose(tabela_escores.loc["sim", "P(C | x)"], 0.892857)
tabela_escores.round(4)

,P(C),P(canal=app | C),P(texto_urgente=sim | C),P(cliente_premium=sim | C),escore(C),P(C | x)
C = prioritária,,,,,,
sim,0.5,0.625,0.75,0.625,0.1465,0.8929
nao,0.5,0.375,0.25,0.375,0.0176,0.1071


### Da contagem à notação probabilística

Na classe prioritária, $C=\text{sim}$:

- quatro dos seis chamados vieram do aplicativo:

$$
P(\text{canal=app}\mid C=\text{sim})
=
\frac{4+1}{6+2}
=
\frac{5}{8};
$$

- cinco dos seis continham texto urgente:

$$
P(\text{texto urgente=sim}\mid C=\text{sim})
=
\frac{5+1}{6+2}
=
\frac{6}{8};
$$

- quatro dos seis eram de clientes premium:

$$
P(\text{cliente premium=sim}\mid C=\text{sim})
=
\frac{4+1}{6+2}
=
\frac{5}{8}.
$$

Portanto, o escore da classe prioritária é:

$$
\begin{aligned}
\operatorname{escore}(C=\text{sim})
&=P(C=\text{sim})\\
&\quad\times P(\text{canal=app}\mid C=\text{sim})\\
&\quad\times P(\text{texto urgente=sim}\mid C=\text{sim})\\
&\quad\times P(\text{cliente premium=sim}\mid C=\text{sim})\\
&=0{,}5\times\frac{5}{8}\times\frac{6}{8}\times\frac{5}{8}\\
&=0{,}146484375\approx0{,}1465.
\end{aligned}
$$

Na classe não prioritária, $C=\text{não}$:

$$
P(\text{canal=app}\mid C=\text{não})
=
\frac{2+1}{6+2}
=
\frac{3}{8},
$$

$$
P(\text{texto urgente=sim}\mid C=\text{não})
=
\frac{1+1}{6+2}
=
\frac{2}{8},
$$

$$
P(\text{cliente premium=sim}\mid C=\text{não})
=
\frac{2+1}{6+2}
=
\frac{3}{8}.
$$

Logo:

$$
\begin{aligned}
\operatorname{escore}(C=\text{não})
&=P(C=\text{não})\\
&\quad\times P(\text{canal=app}\mid C=\text{não})\\
&\quad\times P(\text{texto urgente=sim}\mid C=\text{não})\\
&\quad\times P(\text{cliente premium=sim}\mid C=\text{não})\\
&=0{,}5\times\frac{3}{8}\times\frac{2}{8}\times\frac{3}{8}\\
&=0{,}017578125\approx0{,}0176.
\end{aligned}
$$

### Dos escores às probabilidades posteriores

Os escores ainda não somam 1. Para normalizá-los:

$$
P(C=\text{sim}\mid\mathbf{x})
=
\frac{\operatorname{escore}(C=\text{sim})}
{\operatorname{escore}(C=\text{sim})+\operatorname{escore}(C=\text{não})}
=
\frac{0{,}146484375}{0{,}146484375+0{,}017578125}
=\frac{75}{84}
\approx 0{,}8929,
$$

$$
P(C=\text{não}\mid\mathbf{x})
=
\frac{0{,}017578125}{0{,}146484375+0{,}017578125}
=\frac{9}{84}
\approx 0{,}1071.
$$

Como a primeira posterior é maior, o método prevê **prioritária**.

A posterior depende da base pequena, da suavização e da hipótese de independência. Ela não deve ser interpretada automaticamente como uma probabilidade perfeitamente calibrada no atendimento real.

## 3. Quando considerar Naive Bayes

Naive Bayes é uma referência simples e rápida quando frequências ou distribuições por classe podem ser estimadas. É especialmente útil em texto e contagens esparsas.

A independência condicional pode ser irreal, e evidências correlacionadas podem ser contadas separadamente. Mesmo quando a classe é útil, as posteriores podem exigir avaliação de calibração em dados separados.

## Atividades

> **U05-NB03-V01 — Verifique seu entendimento:** por que dois atributos muito correlacionados podem fazer o Naive Bayes tratar uma evidência como se aparecesse duas vezes? Por que isso não torna necessariamente todas as classificações incorretas?

> **U05-NB03-E01 — Exercício:** usando a tabela e Laplace, classifique $mathbf{x}=(	ext{web}, 	ext{não urgente}, 	ext{não premium})$. Escreva a notação de cada fração, calcule escores, posteriores e classe.

> **U05-NB03-E02 — Laplace e frequência zero:** nenhum chamado não prioritário possui o novo canal “telefone”. Explique o escore sem suavização e calcule esse fator com Laplace se o atributo possuir agora três categorias.

> **U05-NB03-E03 — Decisão de método:** avalie Naive Bayes para milhares de documentos representados por ocorrências de palavras. Justifique por representação e custo e apresente limitações de independência e calibração.

## Síntese

- Bayes combina prévia e verossimilhança.
- Naive Bayes usa um produto de probabilidades condicionais.
- Cada fração deve identificar evento, classe, contagem e suavização.
- Laplace impede que frequência zero anule o produto.
- Escores são normalizados para produzir posteriores.
- Correlação e calibração exigem avaliação crítica.

## Referências

- HAN, J.; PEI, J.; TONG, H. *Data Mining: Concepts and Techniques*. 4. ed. 2023. Seção 6.3.
- SCIKIT-LEARN DEVELOPERS. *User Guide: Naive Bayes*.